In [1]:
import pandas as pd
import numpy as np
import json

output_results = '../results/output.json'
ground_truth = '../ground_truth.csv'

In [2]:
# read json file
with open(output_results, "r") as f:
    results = json.load(f)
output_results = pd.DataFrame(results)
output_results.head()

,image_path,num_balls,balls
0,development_set/106_png.rf.28ee53acf89d9e7f17b...,8,"[{'number': 1, 'xmin': 0.5953125, 'xmax': 0.61..."
1,development_set/10a_png.rf.bdc9984ba169594ea32...,11,"[{'number': 14, 'xmin': 0.38095238095238093, '..."
2,development_set/110_png.rf.9a38b6057e543f83b58...,2,"[{'number': 0, 'xmin': 0.3859375, 'xmax': 0.40..."
3,development_set/114_png.rf.98b2144c25b9f48816a...,14,"[{'number': 0, 'xmin': 0.3380208333333333, 'xm..."
4,development_set/115_png.rf.ed51d53dd5c384b1f26...,11,"[{'number': 6, 'xmin': 0.7182291666666667, 'xm..."


In [3]:
output_results["image_id"] = output_results["image_path"].apply(lambda x: x.split("/")[-1])
output_results = output_results.drop(columns=['image_path'])
output_results.head()

,num_balls,balls,image_id
0,8,"[{'number': 1, 'xmin': 0.5953125, 'xmax': 0.61...",106_png.rf.28ee53acf89d9e7f17b2fb26185597a0.jpg
1,11,"[{'number': 14, 'xmin': 0.38095238095238093, '...",10a_png.rf.bdc9984ba169594ea32b012098ad10dd.jpg
2,2,"[{'number': 0, 'xmin': 0.3859375, 'xmax': 0.40...",110_png.rf.9a38b6057e543f83b58aa59b9748688b.jpg
3,14,"[{'number': 0, 'xmin': 0.3380208333333333, 'xm...",114_png.rf.98b2144c25b9f48816abd7edb00f365c.jpg
4,11,"[{'number': 6, 'xmin': 0.7182291666666667, 'xm...",115_png.rf.ed51d53dd5c384b1f26a2b6ece52ad69.jpg


In [4]:
ground_truth_data = pd.read_csv(ground_truth)
ground_truth_data.head()

,image_id,num_balls,ball_types
0,3f_png.rf.81c7e132365ef95bb19380ca389025f6.jpg,15,"[0,1,2,3,4,5,6,7,8,9,null,11,12,13,14,15]"
1,4a_png.rf.a6bb5c5706fd8628eb53d34a122cf441.jpg,12,"[0,1,2,3,4,5,6,7,8,null,null,null,12,null,14,15]"
2,5a_png.rf.bae1d48b3d2d96a990799b836ecebcbb.jpg,15,"[0,null,2,3,4,5,6,7,8,9,10,11,12,13,14,15]"
3,6f_png.rf.4dcfc0c556af56e418f2226f40765059.jpg,14,"[0,null,2,3,4,5,6,null,8,9,10,11,12,13,14,15]"
4,7a_png.rf.7cf9f36a358969bd16be866d1570a303.jpg,14,"[0,1,2,3,4,5,6,7,8,9,10,11,null,13,14,15]"


In [5]:
eval_df = output_results.merge(
    ground_truth_data,
    on="image_id",
    how="inner",
    suffixes=("_pred", "_gt")
)
eval_df["ball_types"] = eval_df["ball_types"].apply(json.loads)
eval_df.head()

,num_balls_pred,balls,image_id,num_balls_gt,ball_types
0,8,"[{'number': 1, 'xmin': 0.5953125, 'xmax': 0.61...",106_png.rf.28ee53acf89d9e7f17b2fb26185597a0.jpg,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, None, None, Non..."
1,11,"[{'number': 14, 'xmin': 0.38095238095238093, '...",10a_png.rf.bdc9984ba169594ea32b012098ad10dd.jpg,12,"[0, 1, 2, 3, 4, 5, 6, 7, 8, None, 10, 11, None..."
2,2,"[{'number': 0, 'xmin': 0.3859375, 'xmax': 0.40...",110_png.rf.9a38b6057e543f83b58aa59b9748688b.jpg,2,"[0, None, None, None, None, None, None, None, ..."
3,14,"[{'number': 0, 'xmin': 0.3380208333333333, 'xm...",114_png.rf.98b2144c25b9f48816abd7edb00f365c.jpg,16,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
4,11,"[{'number': 6, 'xmin': 0.7182291666666667, 'xm...",115_png.rf.ed51d53dd5c384b1f26a2b6ece52ad69.jpg,11,"[0, None, 2, None, None, None, 6, None, 8, 9, ..."


In [6]:
mae = np.mean(np.abs(eval_df["num_balls_pred"] - eval_df["num_balls_gt"]))
accuracy = np.mean(eval_df["num_balls_pred"] == eval_df["num_balls_gt"])

print(f"Ball Count MAE: {mae:.4f}")
print(f"Ball Count Accuracy: {accuracy:.4f}")


def compute_undetected(gt, pred):
    gt_set = set([b for b in gt if b is not None])
    pred_set = set([b["number"] for b in pred if b["number"] is not None])

    if len(gt_set) == 0:
        return 0

    missed = gt_set - pred_set
    return len(missed) / len(gt_set)


def compute_correct(gt, pred):
    gt_set = set([b for b in gt if b is not None])
    pred_set = set([b["number"] for b in pred if b["number"] is not None])

    if len(gt_set) == 0:
        return 0

    correct = gt_set & pred_set
    return len(correct) / len(gt_set)


def compute_incorrect(gt, pred):
    gt_set = set([b for b in gt if b is not None])
    pred_set = set([b["number"] for b in pred if b["number"] is not None])

    if len(pred_set) == 0:
        return 0

    incorrect = pred_set - gt_set
    return len(incorrect) / len(pred_set)


undetected_list = []
correct_list = []
incorrect_list = []

for _, row in eval_df.iterrows():
    gt = row["ball_types"]
    pred = row["balls"]

    undetected_list.append(compute_undetected(gt, pred))
    correct_list.append(compute_correct(gt, pred))
    incorrect_list.append(compute_incorrect(gt, pred))

print(f"Average % of undetected balls: {np.mean(undetected_list):.4f}")
print(f"Average % of correctly classified balls: {np.mean(correct_list):.4f}")
print(f"Average % of incorrectly classified balls: {np.mean(incorrect_list):.4f}")

Ball Count MAE: 1.6800
Ball Count Accuracy: 0.2400
Average % of undetected balls: 0.4542
Average % of correctly classified balls: 0.5458
Average % of incorrectly classified balls: 0.1492


### Métricas Rodrigo's

In [7]:
def compute_metrics(gt_ball_types, pred_balls):
    """
    Computes classification metrics handling duplicates correctly.
    
    Uses a multiset (Counter) approach:
    - GT:   Counter({3: 1, 5: 1, 8: 1, ...})
    - Pred: Counter({3: 2, 5: 1, 14: 1, ...})
    
    A predicted ball counts as correct only if it hasn't already been
    "consumed" by a previous match to the same GT ball.
    """
    from collections import Counter

    gt_counter   = Counter(b for b in gt_ball_types if b is not None)
    pred_counter = Counter(b["number"] for b in pred_balls if b["number"] is not None)

    # Correctly detected: min(gt_count, pred_count) per ball number
    correct = sum(min(gt_counter[b], pred_counter[b]) for b in gt_counter)

    gt_total   = sum(gt_counter.values())
    pred_total = sum(pred_counter.values())

    # Undetected: GT balls not matched by any prediction
    undetected = (gt_total - correct) / gt_total if gt_total > 0 else 0

    # Correctly classified: fraction of GT balls that were matched
    correctly_classified = correct / gt_total if gt_total > 0 else 0

    # Incorrectly classified: fraction of predictions that didn't match any GT
    incorrectly_classified = (pred_total - correct) / pred_total if pred_total > 0 else 0

    return undetected, correctly_classified, incorrectly_classified


# --- Recalcular métricas ---
mae      = np.mean(np.abs(eval_df["num_balls_pred"] - eval_df["num_balls_gt"]))
accuracy = np.mean(eval_df["num_balls_pred"] == eval_df["num_balls_gt"])
print(f"Ball Count MAE:      {mae:.4f}")
print(f"Ball Count Accuracy: {accuracy:.4f}")

undetected_list   = []
correct_list      = []
incorrect_list    = []

for _, row in eval_df.iterrows():
    u, c, i = compute_metrics(row["ball_types"], row["balls"])
    undetected_list.append(u)
    correct_list.append(c)
    incorrect_list.append(i)

print(f"Undetected balls:            {np.mean(undetected_list):.4f}")
print(f"Correctly classified balls:  {np.mean(correct_list):.4f}")
print(f"Incorrectly classified balls:{np.mean(incorrect_list):.4f}")

Ball Count MAE:      1.6800
Ball Count Accuracy: 0.2400
Undetected balls:            0.4542
Correctly classified balls:  0.5458
Incorrectly classified balls:0.3551
